## 24_Features_Digital — NBA 3.1 (Family 1: HCP paid media)

NBA 3.1 rewrite of the NBA 3.0 notebook `03_HCP_Level_Features/24_Features_Digital`, following section 3.1 (Family 1) of `52_digital_feature_families_nba30_vs_nba31.md`.

**What changes vs NBA 3.0**

| Topic | NBA 3.0 (`24_Features_Digital`) | NBA 3.1 (this notebook) |
| --- | --- | --- |
| Source | Snowflake `MMX_FNL_COMBINED_AGGR_TAB` | UC copy `kerendia_oe_journey.mmx_fnl_combined_aggr_tab` |
| Brand filter | `UPPER(BRND_NM) = 'KERENDIA'` (dropped 88.5% Doceree, 86.5% ReachMD, 67.3% PeerDirect) | `UPPER(BRND_NM) LIKE '%KERENDIA%'`; the indication (HF / CKD) in the brand value becomes a dimension |
| Scope | Calls / samples removed, but HQ email and iRep email kept as channels and vendors | Only paid media (banner, 3rd-party email, custom). Calls, samples, HQ email, iRep email excluded (own families) |
| Mapping | Exact tactic names; renamed tactics fell into `OTHER_TACTIC` | Fixed placement dictionary (regex on `UPPER(TRIM())`), every vendor on its own, unmapped report each run |
| Schema | 3 dynamic pivots (channel, vendor, tactic) x 4 measures x 1/3/6 months + `RUNNING_`: 684 columns that change between runs | Fixed list generated from config: totals, channels, placements, rates, recency, diversity, trend, indication. No `RUNNING_` features |
| Doctor key / grain | `NPI_NUMBER` x month, single inference cohort I01 | `PROVIDER_ID` (NBA 3.1 HCP universe) x `COHORT` (c1 – c28 from the NBA 3.1 promotional cohorts file) |
| Windows | Calendar months incl. the unfinished current month | 1 / 3 / 6 / 12 months ending at `LOOKBACK_END_DATE` (inclusive), i.e. `(END - w months, END]` |
| NPI quality | `PDRP` / `'NULL'` NPIs counted as doctors | Only 10-digit NPIs |

**Inputs**
* HCP universe: `kerendia_nba_3_1_volume/HCP_Universe_without_call_hcps/` (`PROVIDER_ID`, placeholder `0` removed)
* Cohorts: `kerendia_nba_3_1_volume/cohorts/promotional/cohorts.parquet` (28 cohorts, offset 17 days)
* NPI → PROVIDER_ID: `kerendia_nba_3.SOURCE_PROVIDER_BHID_NPI_MAPPING`

**Data-history caveat:** MMX paid media starts in January 2025. Cohorts whose `LOOKBACK_END_DATE` is before 2025 (c1 – c7) have no digital data, and windows are only complete when `DIGITAL_HISTORY_MONTHS` ≥ window length. Trend features are NULL when history < 6 months.

In [0]:
import re, json, os, hashlib
from datetime import datetime
import pandas as pd
import pyspark.sql.functions as F
from pyspark.sql.functions import col, when, lit

# ---------- Sources ----------
CATALOG       = "ph-com-ai-us-prod-catalog-2553213839599380"
MMX_TABLE     = f"`{CATALOG}`.kerendia_oe_journey.mmx_fnl_combined_aggr_tab"
MAPPING_TABLE = f"`{CATALOG}`.kerendia_nba_3.SOURCE_PROVIDER_BHID_NPI_MAPPING"   # PROVIDER_ID, NPI_NUMBER, BH_ID

V31           = "/Volumes/ph-com-ai-us-prod-catalog-2553213839599380/kerendia_nba_3/kerendia_nba_3_1_volume"
COHORT_PATH   = f"{V31}/cohorts/promotional/cohorts.parquet"
UNIVERSE_PATH = f"{V31}/HCP_Universe_without_call_hcps/"

# ---------- Outputs ----------
OUT_DIR       = f"{V31}/notebook_output/digital_engagement_nba_3_1"
OUTPUT_PATH   = f"{OUT_DIR}/kerendia_nba_3_1_hcp_model_digital_paid_media_features.parquet"
STAGE_PATH    = f"{OUT_DIR}/_stage_paid_media_monthly.parquet"   # PROVIDER_ID x cohort-aligned month
DICT_PATH     = f"{OUT_DIR}/placement_dictionary.parquet"
UNMAPPED_PATH = f"{OUT_DIR}/unmapped_report.parquet"           # appended each run
MANIFEST_PATH = f"{OUT_DIR}/feature_manifest.parquet"
SCHEMA_PATH   = f"{OUT_DIR}/expected_schema.json"

# ---------- Rules ----------
DIGITAL_DATA_START   = "2025-01-01"   # first MMX paid-media row in the UC copy
WINDOWS              = [1, 3, 6, 12]  # months, ending at LOOKBACK_END_DATE (inclusive)
RECENCY_SENTINEL     = 999            # days since last event when none in the last 12 months (same as field notebook 41)
TREND_MIN_HISTORY    = 6              # trend features NULL when fewer complete months of digital history
UNMAPPED_ALERT_SHARE = 0.01
SAMPLE_PROVIDERS     = None           # e.g. ["10000012", ...] for a quick test run

os.makedirs(OUT_DIR, exist_ok=True)
print(f"Output: {OUTPUT_PATH}")

In [0]:
# ---------- NBA 3.1 promotional cohorts ----------
cohorts = (spark.read.parquet(COHORT_PATH)
           .select(col("cohort_id").alias("COHORT"), col("dataset").alias("DATASET"),
                   col("prediction_month").alias("PREDICTION_MONTH"), col("offset_days").alias("OFFSET_DAYS"),
                   col("lookback_start_date").alias("LOOKBACK_START_DATE"),
                   col("lookback_end_date").alias("LOOKBACK_END_DATE")))
cohort_pd = cohorts.toPandas()
cohort_pd["_N"] = cohort_pd.COHORT.str[1:].astype(int)
cohort_pd = cohort_pd.sort_values("_N").drop(columns="_N").reset_index(drop=True)

# Windows are aligned on the day of LOOKBACK_END_DATE (17 for every cohort). A "cohort month" bucket B covers
# (B + END_DAY days, B + 1 month + END_DAY days], so the w-month window (END - w months, END] = the last w buckets.
_end_days = set(pd.to_datetime(cohort_pd.LOOKBACK_END_DATE).dt.day)
assert len(_end_days) == 1, f"Cohorts do not share one end day: {_end_days}"
END_DAY = _end_days.pop()

def bucket_col(c):
    return F.trunc(F.date_sub(F.to_date(c), END_DAY), "month")

# Self-test of the bucket rule (END_DAY = 17): 2026-09-17 -> Aug, 2026-09-18 -> Sep, 2026-08-17 -> Jul
if END_DAY == 17:
    _t = spark.createDataFrame([("2026-09-17", "2026-08-01"), ("2026-09-18", "2026-09-01"), ("2026-08-17", "2026-07-01"),
                                ("2025-01-01", "2024-12-01"), ("2025-01-18", "2025-01-01")], "D string, EXP string")
    _bad = _t.filter(bucket_col(col("D")).cast("string") != col("EXP")).count()
    assert _bad == 0, "bucket rule self-test failed"

# First complete bucket of digital data and history per cohort (capped at 12)
FIRST_COMPLETE_BUCKET = spark.range(1).select(
    F.add_months(bucket_col(lit(DIGITAL_DATA_START)), 1).alias("B")).first()["B"]   # data starts on the 1st -> first bucket partial
cohorts = (cohorts.withColumn("END_BUCKET", bucket_col(col("LOOKBACK_END_DATE")))
           .withColumn("DIGITAL_HISTORY_MONTHS",
                       F.least(lit(12), F.greatest(lit(0), F.months_between(col("END_BUCKET"), lit(FIRST_COMPLETE_BUCKET)).cast("int") + 1))))
print(f"{len(cohort_pd)} cohorts {cohort_pd.COHORT.iloc[0]} .. {cohort_pd.COHORT.iloc[-1]}, END_DAY = {END_DAY}, "
      f"first complete digital bucket = {FIRST_COMPLETE_BUCKET}")
display(cohorts.orderBy(F.expr("int(substring(COHORT, 2))")))

# ---------- HCP universe (PROVIDER_ID = IQVIA id) ----------
universe = spark.read.parquet(UNIVERSE_PATH).select("PROVIDER_ID").distinct()
n_before = universe.count()
universe = universe.filter(col("PROVIDER_ID").isNotNull() & (col("PROVIDER_ID") != "0"))   # placeholder
if SAMPLE_PROVIDERS:
    universe = universe.filter(col("PROVIDER_ID").isin(SAMPLE_PROVIDERS))
N_UNIVERSE = universe.count()
print(f"Universe: {N_UNIVERSE:,} PROVIDER_IDs (removed {n_before - N_UNIVERSE:,})")

# ---------- NPI -> PROVIDER_ID (universe only, 10-digit NPIs) ----------
npi_map_all = (spark.table(MAPPING_TABLE)
               .select("PROVIDER_ID", F.trim("NPI_NUMBER").alias("NPI_NUMBER"))
               .filter(col("PROVIDER_ID").isNotNull() & col("NPI_NUMBER").rlike("^[0-9]{10}$"))
               .distinct()
               .join(universe, "PROVIDER_ID", "left_semi"))
n_npi_multi = npi_map_all.groupBy("NPI_NUMBER").count().filter("count > 1").count()
# An NPI on more than one PROVIDER_ID goes to the smallest PROVIDER_ID, so each exposure is counted once
npi_map = npi_map_all.groupBy("NPI_NUMBER").agg(F.min("PROVIDER_ID").alias("PROVIDER_ID"))
print(f"Universe PROVIDER_IDs with an NPI: {npi_map_all.select('PROVIDER_ID').distinct().count():,}; "
      f"NPIs on >1 PROVIDER_ID: {n_npi_multi} (smallest PROVIDER_ID kept)")

In [0]:
# Same dictionary as 51_digital_nba_phase1_features. Regexes are full matches on UPPER(TRIM()) values with whitespace
# collapsed; the first matching rule wins. Rows that match no rule are PAID_MEDIA_UNMAPPED (reported below).
PM = "PAID_MEDIA"
DICT_FIELDS = ["RULE_ID", "CHANNEL_RAW", "VENDOR_RAW", "TACTIC_RAW", "ROLE", "CHANNEL", "VENDOR",
               "PLACEMENT", "OE_PLACEMENT", "OE_RECOMMENDS", "NOTE"]
_PD = "(MDFI|MD MINUTE|MD SHUFFLE|PV COMPENDIUM).*"
DICT_RULES = [dict(zip(DICT_FIELDS, r)) for r in [
    # ---- Exclusions (not family 1) ----
    ("X01", "CALLS?",      ".*", ".*", "EXCLUDE_FIELD", None, None, None, None, None, "Rep calls: personal block"),
    ("X02", "SAMPLES?",    ".*", ".*", "EXCLUDE_FIELD", None, None, None, None, None, "Samples: own notebook"),
    ("X03", "IREP ?EMAIL", ".*", ".*", "EXCLUDE_IREP", None, None, None, None, None, "iRep: personal block"),
    ("X04", ".*", "IREP ?EMAIL", ".*", "EXCLUDE_IREP", None, None, None, None, None, "iRep"),
    ("X05", ".*", ".*", "IREP ?EMAIL", "EXCLUDE_IREP", None, None, None, None, None, "iRep"),
    ("X06", "HQ ?EMAIL",   ".*", ".*", "EXCLUDE_HQ_EMAIL", None, None, None, "SFMC_HQ_EMAIL", "Y", "HQ email: family 2"),
    # ---- Banner ----
    ("B01", "BANNER", "PULSE ?POINT", "PROGRAM+ATIC[ _]?DISPLAY", PM, "BANNER", "PULSEPOINT", "PULSEPOINT_PROGRAMMATIC_DISPLAY", "PULSEPOINT_BANNER", "Y", "About 98% of banner volume"),
    ("B02", "BANNER", "MEDSCAPE", "|BRAND ALERT|DIGITAL DATA TRENDS", PM, "BANNER", "MEDSCAPE", "MEDSCAPE_BANNER", "MEDSCAPE_BANNER", "Y", ""),
    ("B03", "BANNER", "DOCEREE", "EHR ?BANNER", PM, "BANNER", "DOCEREE", "DOCEREE_EHR_BANNER", None, "N", "Started April 2025"),
    ("B04", "BANNER", "PEER ?DIRECT", _PD, PM, "BANNER", "PEERDIRECT", "PEERDIRECT_BANNER", None, "N", "Started April 2025"),
    ("B05", "BANNER", "OPEN ?EVIDENCE", "DIRECT ?BANNER", PM, "BANNER", "OPENEVIDENCE", "OPENEVIDENCE_DIRECT_BANNER", None, "N", "Started April 2025"),
    ("B06", "BANNER", "BIOPHARM.*", ".*", PM, "BANNER", "BIOPHARM", "BIOPHARM_BANNER", None, "N", ""),
    ("B07", "BANNER", "HAYMARKET.*", ".*", PM, "BANNER", "HAYMARKET", "HAYMARKET_BANNER", None, "N", "Raw tactic = campaign name"),
    ("B08", "BANNER", "REACH ?MD", ".*", PM, "BANNER", "REACHMD", "REACHMD_BANNER", None, "N", ""),
    ("B09", "BANNER", "CONNECTIVE ?RX", ".*", PM, "BANNER", "CONNECTIVERX", "CONNECTIVERX_BANNER", None, "N", "Vendor code to confirm; exposure only"),
    ("B10", "BANNER", "NATIVO", ".*", PM, "BANNER", "NATIVO", "NATIVO_BANNER", None, "N", "Vendor code to confirm"),
    ("B11", "BANNER", "HEALTH ?CAST", ".*", PM, "BANNER", "HEALTHCAST", "HEALTHCAST_BANNER", None, "N", "Vendor code to confirm"),
    ("B12", "BANNER", "MJH.*", ".*", PM, "BANNER", "MJH", "MJH_BANNER", None, "N", ""),
    ("B13", "BANNER", "LINKEDIN", ".*", PM, "BANNER", "LINKEDIN", "LINKEDIN_BANNER", None, "N", "New in data; review"),
    # ---- 3rd-party email ----
    ("E01", "3RD PARTY EMAIL", "MEDSCAPE", "|EMAIL ALERT|FORMULARY ACCESS PACKAGE|DIGITAL DATA TRENDS", PM, "3P_EMAIL", "MEDSCAPE", "MEDSCAPE_EMAIL_ALERT", "MEDSCAPE_3RD_PARTY_EMAIL", "Y", ""),
    ("E02", "3RD PARTY EMAIL", "PEER ?DIRECT", "MDFI.*", PM, "3P_EMAIL", "PEERDIRECT", "PEERDIRECT_MDFI_EMAIL", "PEERDIRECT_MDFI_EMAIL", "Y", ""),
    ("E03", "3RD PARTY EMAIL", "PEER ?DIRECT", "MD MINUTE.*", PM, "3P_EMAIL", "PEERDIRECT", "PEERDIRECT_MD_MINUTE_EMAIL", "PEERDIRECT_MD_MINUTE_EMAIL", "Y", ""),
    ("E04", "3RD PARTY EMAIL", "HAYMARKET.*", ".*", PM, "3P_EMAIL", "HAYMARKET", "HAYMARKET_3P_EMAIL", None, "N", "Raw tactic = campaign name"),
    ("E05", "3RD PARTY EMAIL", "BIOPHARM.*", ".*", PM, "3P_EMAIL", "BIOPHARM", "BIOPHARM_3P_EMAIL", None, "N", ""),
    ("E06", "3RD PARTY EMAIL", "MJH.*", ".*", PM, "3P_EMAIL", "MJH", "MJH_3P_EMAIL", None, "N", ""),
    ("E07", "3RD PARTY EMAIL", "HEALTH ?CAST", ".*", PM, "3P_EMAIL", "HEALTHCAST", "HEALTHCAST_3P_EMAIL", None, "N", "Vendor code to confirm"),
    # ---- Custom (medical sites and apps) ----
    ("C01", "CUSTOM", "DOXIMITY", "DOC ?NEWS", PM, "CUSTOM", "DOXIMITY", "DOXIMITY_DOCNEWS", "DOXIMITY_DOCNEWS", "Y", ""),
    ("C02", "CUSTOM", "DOXIMITY", "DOC ?POINT", PM, "CUSTOM", "DOXIMITY", "DOXIMITY_DOCPOINT", "DOXIMITY_DOCPOINT", "Y", ""),
    ("C03", "CUSTOM", "DOXIMITY", "DOC ?SPOT", PM, "CUSTOM", "DOXIMITY", "DOXIMITY_DOCSPOT", "DOXIMITY_DOCSPOT", "Y", ""),
    ("C04", "CUSTOM", "DOXIMITY", "ANNOUNCEMENT.*", PM, "CUSTOM", "DOXIMITY", "DOXIMITY_ANNOUNCEMENT", "DOXIMITY_ANNOUNCEMENT", "Y", ""),
    ("C05", "CUSTOM", "DOXIMITY", "FORMULARY ?ONE|FORMULARY", PM, "CUSTOM", "DOXIMITY", "DOXIMITY_FORMULARYONE", None, "N", ""),
    ("C06", "CUSTOM", "MEDSCAPE", "(BA360 - )?BRAND ?ALERT", PM, "CUSTOM", "MEDSCAPE", "MEDSCAPE_BRANDALERT", "MEDSCAPE_BRANDALERT", "Y", ""),
    ("C07", "CUSTOM", "MEDSCAPE", "RESEARCH CAPSULE.*|FAST FIVE QUIZ|BRANDSPOT|XC ON DEMAND", PM, "CUSTOM", "MEDSCAPE", "MEDSCAPE_EDUCATION", None, "N", "ReCAP, FFQ, BrandSpot, XCOD"),
    ("C08", "CUSTOM", "MEDSCAPE", "|OFFSITE MEDIA|MEDIA|BA360 - EMAIL ALERT|DIGITAL DATA TRENDS|FORMULARY ACCESS PACKAGE", PM, "CUSTOM", "MEDSCAPE", "MEDSCAPE_OFFSITE_MEDIA", None, "N", "Review"),
    ("C09", "CUSTOM", "EPOCRATES", "MONOGRAPH.*", PM, "CUSTOM", "EPOCRATES", "EPOCRATES_MONOGRAPH", None, "N", ""),
    ("C10", "CUSTOM", "EPOCRATES", "DOC ?ALERT", PM, "CUSTOM", "EPOCRATES", "EPOCRATES_DOCALERT", None, "N", ""),
    ("C11", "CUSTOM", "EPOCRATES", "SEARCH SUGGESTION.*", PM, "CUSTOM", "EPOCRATES", "EPOCRATES_SEARCH_SUGGESTION", None, "N", ""),
    ("C12", "CUSTOM", "PRESCRIBER[ _]?POINT", "MONOGRAPH.*", PM, "CUSTOM", "PRESCRIBERPOINT", "PRESCRIBERPOINT_MONOGRAPH", None, "N", ""),
    ("C13", "CUSTOM", "PEER ?DIRECT", _PD, PM, "CUSTOM", "PEERDIRECT", "PEERDIRECT_WEB", "PEERDIRECT_MDFI_WEB", "Y", ""),
    ("C14", "CUSTOM", "IMPIRICUS", "SMS.*", PM, "CUSTOM", "IMPIRICUS", "IMPIRICUS_SMS", None, "N", "No MMX rows after Dec 2025"),
    ("C15", "CUSTOM", "REACH ?MD", ".*", PM, "CUSTOM", "REACHMD", "REACHMD_CUSTOM", None, "N", ""),
    ("C16", "CUSTOM", "MD[ _]?CALC", ".*", PM, "CUSTOM", "MDCALC", "MDCALC_CUSTOM", None, "N", ""),
    ("C17", "CUSTOM", "HAYMARKET.*", ".*", PM, "CUSTOM", "HAYMARKET", "HAYMARKET_CUSTOM", None, "N", ""),
    ("C18", "CUSTOM", "MJH.*", ".*", PM, "CUSTOM", "MJH", "MJH_CUSTOM", None, "N", ""),
]]
CHANNELS   = ["BANNER", "3P_EMAIL", "CUSTOM"]
PLACEMENTS = list(dict.fromkeys(r["PLACEMENT"] for r in DICT_RULES if r["ROLE"] == PM))
VENDORS    = list(dict.fromkeys(r["VENDOR"] for r in DICT_RULES if r["ROLE"] == PM))
RAW_CH_TO_TOKEN = {"BANNER": "BANNER", "3RD PARTY EMAIL": "3P_EMAIL", "CUSTOM": "CUSTOM"}
PP = "PULSEPOINT_PROGRAMMATIC_DISPLAY"   # XPP features = everything except this placement

def normalize(s):
    return re.sub(r"\s+", " ", (s or "").strip().upper()).strip()

def map_combo(ch, v, t):
    ch, v, t = normalize(ch), normalize(v), normalize(t)
    for r in DICT_RULES:
        if re.fullmatch(r["CHANNEL_RAW"], ch) and re.fullmatch(r["VENDOR_RAW"], v) and re.fullmatch(r["TACTIC_RAW"], t):
            return {"RULE_ID": r["RULE_ID"], "ROLE": r["ROLE"], "CHANNEL": r["CHANNEL"], "VENDOR": r["VENDOR"], "PLACEMENT": r["PLACEMENT"]}
    return {"RULE_ID": "UNMAPPED", "ROLE": "PAID_MEDIA_UNMAPPED", "CHANNEL": RAW_CH_TO_TOKEN.get(ch, "OTHER"),
            "VENDOR": re.sub(r"[^A-Z0-9]+", "_", v).strip("_") or "UNKNOWN", "PLACEMENT": "OTHER"}

# Self-test: known raw strings, including the NBA 3.0 failures (case, spelling, empty tactic, renamed tactic)
_cases = [
    (("IREP EMAIL", "IREP EMAIL", "iRep Email"), "EXCLUDE_IREP", None),
    (("HQ Email", "SFMC", "HQ Email"), "EXCLUDE_HQ_EMAIL", None),
    (("Call", "CALLS", ""), "EXCLUDE_FIELD", None),
    (("Sample", "SAMPLES", ""), "EXCLUDE_FIELD", None),
    (("Banner", "PulsePoint", "Programatic_DISPLAY"), PM, PP),
    (("BANNER", "PULSEPOINT", "Programmatic  Display"), PM, PP),
    (("CUSTOM", "MEDSCAPE", "BA360 - Brand Alert"), PM, "MEDSCAPE_BRANDALERT"),
    (("Custom", "Doximity", "DocSpot"), PM, "DOXIMITY_DOCSPOT"),
    (("Custom", "Doximity", "Doc News"), PM, "DOXIMITY_DOCNEWS"),
    (("3rd Party Email", "Medscape", ""), PM, "MEDSCAPE_EMAIL_ALERT"),
    (("Custom", "MD_CALC", "Anything"), PM, "MDCALC_CUSTOM"),
    (("Banner", "Some New Vendor", "Foo"), "PAID_MEDIA_UNMAPPED", "OTHER"),
]
for (ch, v, t), role, plc in _cases:
    m = map_combo(ch, v, t)
    assert m["ROLE"] == role and (plc is None or m["PLACEMENT"] == plc), ((ch, v, t), m)

DICT_VERSION = hashlib.md5(json.dumps(DICT_RULES, sort_keys=True).encode()).hexdigest()[:10]
dict_pdf = pd.DataFrame(DICT_RULES).assign(DICT_VERSION=DICT_VERSION)
(spark.createDataFrame(dict_pdf.astype(object).where(dict_pdf.notna(), None).values.tolist(),
                       ", ".join(f"{c} string" for c in dict_pdf.columns))
      .write.mode("overwrite").parquet(DICT_PATH))
print(f"Dictionary {DICT_VERSION}: {len(DICT_RULES)} rules, {len(PLACEMENTS)} placements, {len(VENDORS)} vendors; "
      f"self-test {len(_cases)}/{len(_cases)} OK")

In [0]:
def _norm(c):
    return F.trim(F.upper(F.regexp_replace(F.coalesce(col(c).cast("string"), lit("")), r"\s+", " ")))

mmx = (spark.table(MMX_TABLE)
       .filter(F.upper(col("BRND_NM")).like("%KERENDIA%"))          # NBA 3.0 used = 'KERENDIA'
       .filter(col("DATE_ACT") >= F.lit(DIGITAL_DATA_START).cast("date"))
       .withColumns({"CH_N": _norm("CHANNEL"), "V_N": _norm("SRC_SYS_CD"), "T_N": _norm("TACTIC_NAME"),
                     "NPI_NUMBER": F.trim(col("NPI_NUM"))})
       .withColumn("NPI_OK", F.coalesce(col("NPI_NUMBER").rlike("^[0-9]{10}$"), lit(False))))   # drops 'PDRP', 'NULL'

combos = (mmx.groupBy("CH_N", "V_N", "T_N").agg(
              F.count("*").alias("ROWS"),
              F.sum(col("NPI_OK").cast("long")).alias("ROWS_VALID_NPI"),
              F.sum(when(col("NPI_OK"), col("EXPOSURE"))).cast("double").alias("EXPOSURE"),
              F.sum(when(col("NPI_OK"), col("TOTAL_ENGAGEMENT"))).cast("double").alias("ENGAGEMENT"),
              F.min("DATE_ACT").alias("MIN_DT"), F.max("DATE_ACT").alias("MAX_DT"))
          .toPandas())
combos = pd.concat([combos, combos.apply(lambda r: pd.Series(map_combo(r.CH_N, r.V_N, r.T_N)), axis=1)], axis=1)
combos[["EXPOSURE", "ENGAGEMENT"]] = combos[["EXPOSURE", "ENGAGEMENT"]].fillna(0.0)

_map_cols = ["CH_N", "V_N", "T_N", "RULE_ID", "ROLE", "CHANNEL", "VENDOR", "PLACEMENT"]
_mp = combos[_map_cols].astype(object)
mapping_sdf = spark.createDataFrame(_mp.where(_mp.notna(), None).values.tolist(),
                                    ", ".join(f"{c} string" for c in _map_cols))

print("Rows by role (all Kerendia brand values):")
display(combos.groupby("ROLE")[["ROWS", "ROWS_VALID_NPI", "EXPOSURE", "ENGAGEMENT"]].sum().reset_index())

# ---- Unmapped report (paid-media rows only) ----
pm_c = combos[combos.ROLE.isin([PM, "PAID_MEDIA_UNMAPPED"])]
unm  = pm_c[pm_c.ROLE == "PAID_MEDIA_UNMAPPED"]
share_rows = float(unm.ROWS_VALID_NPI.sum() / max(pm_c.ROWS_VALID_NPI.sum(), 1))
share_exp  = float(unm.EXPOSURE.sum() / max(pm_c.EXPOSURE.sum(), 1))
ALERT = share_rows > UNMAPPED_ALERT_SHARE or share_exp > UNMAPPED_ALERT_SHARE
print(f"Unmapped share: rows {share_rows:.4%}, exposure {share_exp:.4%} -> {'ALERT' if ALERT else 'OK'}")
if len(unm):
    display(unm.sort_values("ROWS_VALID_NPI", ascending=False))
(spark.createDataFrame([(datetime.now(), DICT_VERSION, int(pm_c.ROWS_VALID_NPI.sum()), int(unm.ROWS_VALID_NPI.sum()),
                         share_rows, share_exp, bool(ALERT),
                         unm[["CH_N", "V_N", "T_N", "ROWS_VALID_NPI", "EXPOSURE"]].to_json(orient="records"))],
                       "RUN_TS timestamp, DICT_VERSION string, PM_ROWS long, UNMAPPED_ROWS long, SHARE_ROWS double, "
                       "SHARE_EXPOSURE double, ALERT boolean, UNMAPPED_COMBOS string")
      .write.mode("append").parquet(UNMAPPED_PATH))

# ---- Data end: earliest last date over the three paid-media channels ----
_ends = pm_c[pm_c.ROLE == PM].groupby("CHANNEL").MAX_DT.max()
PM_DATA_END = min(_ends.values)
print(f"Paid-media last date per channel: {_ends.to_dict()} -> common end {PM_DATA_END}")
_late = cohort_pd[pd.to_datetime(cohort_pd.LOOKBACK_END_DATE) > pd.Timestamp(PM_DATA_END)]
assert _late.empty, f"Cohorts end after the paid-media data end: {_late.COHORT.tolist()}"

display(pm_c.groupby(["CHANNEL", "VENDOR", "PLACEMENT"], dropna=False)[["ROWS_VALID_NPI", "EXPOSURE", "ENGAGEMENT"]]
            .sum().reset_index().sort_values(["CHANNEL", "EXPOSURE"], ascending=[True, False]))

In [0]:
# Indication from the brand value (+ CF_INDICATION when present), as in notebook 51
_ind_src = [col("BRND_NM")] + ([col("CF_INDICATION")] if "CF_INDICATION" in mmx.columns else [])
_ind = F.upper(F.concat_ws(" ", *_ind_src))
INDICATIONS = ["HF", "CKD", "BOTH"]

pm_rows = (mmx.filter(col("NPI_OK") & (col("DATE_ACT") <= F.lit(PM_DATA_END)))
           .drop(*[c for c in ["CHANNEL", "VENDOR", "PLACEMENT", "RULE_ID", "ROLE"] if c in mmx.columns])  # avoid ambiguity
           .join(F.broadcast(mapping_sdf.filter(col("ROLE").isin(PM, "PAID_MEDIA_UNMAPPED"))), ["CH_N", "V_N", "T_N"])
           .join(F.broadcast(npi_map), "NPI_NUMBER")                               # universe HCPs only
           .withColumn("_HF", _ind.rlike("HF|FINEARTS"))
           .withColumn("_CKD", _ind.rlike("CKD|UACR|FINEONE"))
           .withColumn("INDICATION", when(col("_HF") & col("_CKD"), "BOTH").when(col("_HF"), "HF")
                                     .when(col("_CKD"), "CKD").otherwise("GENERAL"))
           .withColumn("B", bucket_col(col("DATE_ACT")))
           .withColumn("EXP", F.coalesce(col("EXPOSURE").cast("double"), lit(0.0)))
           .withColumn("ENG", F.coalesce(col("TOTAL_ENGAGEMENT").cast("double"), lit(0.0))))

# Step 1: long table PROVIDER_ID x B x channel x vendor x placement x indication
long = (pm_rows.groupBy("PROVIDER_ID", "B", "CHANNEL", "VENDOR", "PLACEMENT", "INDICATION")
        .agg(F.sum("EXP").alias("EXP"), F.sum("ENG").alias("ENG"),
             F.max(when(col("EXP") > 0, col("DATE_ACT"))).alias("LAST_EXP_DT"),
             F.max(when(col("ENG") > 0, col("DATE_ACT"))).alias("LAST_ENG_DT")))

# Step 2: wide table with a fixed column list (no dynamic pivot)
def csum(cond, x):
    return F.sum(when(cond, x).otherwise(0.0))
exp, eng = col("EXP"), col("ENG")
xpp = col("PLACEMENT") != PP
is_ch = {ch: col("CHANNEL") == ch for ch in CHANNELS}
is_ch["OTHER"] = ~col("CHANNEL").isin(CHANNELS)

stage_aggs = [F.sum(exp).alias("EXP_ALL"), F.sum(eng).alias("ENG_ALL"),
              csum(xpp, exp).alias("EXP_XPP"), csum(xpp, eng).alias("ENG_XPP"),
              F.max("LAST_EXP_DT").alias("LAST_EXP_DT"), F.max("LAST_ENG_DT").alias("LAST_ENG_DT")]
for ch, cond in is_ch.items():
    stage_aggs += [csum(cond, exp).alias(f"EXP_CH_{ch}"), csum(cond, eng).alias(f"ENG_CH_{ch}")]
for ch in ["3P_EMAIL", "CUSTOM"]:
    stage_aggs.append(F.max(when(is_ch[ch], col("LAST_ENG_DT"))).alias(f"LAST_ENG_DT_{ch}"))
for p in PLACEMENTS:
    c = col("PLACEMENT") == p
    stage_aggs += [csum(c, exp).alias(f"EXP_PLC_{p}"), csum(c, eng).alias(f"ENG_PLC_{p}")]
for v in VENDORS:
    stage_aggs.append(csum((col("VENDOR") == v) & (col("PLACEMENT") != "OTHER"), exp + eng).alias(f"ACT_V_{v}"))
for i in INDICATIONS:
    c = col("INDICATION") == i
    stage_aggs += [csum(c, exp).alias(f"EXP_IND_{i}"), csum(c, eng).alias(f"ENG_IND_{i}")]

(long.groupBy("PROVIDER_ID", "B").agg(*stage_aggs)
     .write.mode("overwrite").parquet(STAGE_PATH))
stage = spark.read.parquet(STAGE_PATH)

print(f"Stage rows (PROVIDER_ID x month): {stage.count():,}; providers with paid media: "
      f"{stage.select('PROVIDER_ID').distinct().count():,} of {N_UNIVERSE:,}")
# Check: channels add up to the total in every row
_diff = stage.select(F.max(F.abs(col("EXP_ALL") - sum(col(f"EXP_CH_{c}") for c in is_ch))).alias("d_exp"),
                     F.max(F.abs(col("ENG_ALL") - sum(col(f"ENG_CH_{c}") for c in is_ch))).alias("d_eng")).first()
assert _diff.d_exp < 1e-6 and _diff.d_eng < 1e-6, _diff
display(stage.groupBy("B").agg(F.countDistinct("PROVIDER_ID").alias("PROVIDERS"), F.sum("EXP_ALL").alias("EXPOSURE"),
                               F.sum("ENG_ALL").alias("ENGAGEMENT"), F.sum("EXP_XPP").alias("EXPOSURE_XPP"),
                               F.sum("EXP_CH_OTHER").alias("EXPOSURE_OTHER_CHANNEL")).orderBy("B"))

In [0]:
METRICS = {"EXPOSURE": "EXP", "ENGAGEMENT": "ENG"}
CH_LABEL = {"BANNER": "banner", "3P_EMAIL": "3rd-party email", "CUSTOM": "custom (medical sites / apps)"}
PLACEMENT_FEATURES  = [("EXPOSURE", 3), ("ENGAGEMENT", 6)]
INDICATION_FEATURES = [("EXPOSURE", 3), ("EXPOSURE", 6), ("ENGAGEMENT", 6)]
RATE_SPECS = [(None, 3), (None, 6), (None, 12), ("XPP", 6), ("BANNER", 6), ("3P_EMAIL", 6), ("CUSTOM", 6)]

SPECS = []
def add(name, family, kind, desc, window=None):
    SPECS.append(dict(FEATURE=name, FAMILY=family, KIND=kind, WINDOW_MONTHS=window, DESCRIPTION=desc))

for w in WINDOWS:                                                     # P1 totals
    for m in METRICS:
        add(f"DIGITAL_TOTAL_{m}_{w}M", "P1", "COUNT", f"All HCP paid media, {m.lower()}, last {w} months", w)
        add(f"DIGITAL_TOTAL_XPP_{m}_{w}M", "P1", "COUNT", f"HCP paid media without PulsePoint programmatic display, {m.lower()}, last {w} months", w)
for w in WINDOWS:                                                     # P2 channel totals
    for ch in CHANNELS:
        for m in METRICS:
            add(f"DIGITAL_CH_{ch}_{m}_{w}M", "P2", "COUNT", f"{CH_LABEL[ch]}, {m.lower()}, last {w} months", w)
for p in PLACEMENTS:                                                  # P3 vendor x placement
    for m, w in PLACEMENT_FEATURES:
        add(f"DIGITAL_PLC_{p}_{m}_{w}M", "P3", "COUNT", f"{p}, {m.lower()}, last {w} months", w)
for ch, w in RATE_SPECS:                                              # P4 engagement rates
    nm = (f"DIGITAL_ENGAGEMENT_RATE_{w}M" if ch is None else
          f"DIGITAL_XPP_ENGAGEMENT_RATE_{w}M" if ch == "XPP" else f"DIGITAL_CH_{ch}_ENGAGEMENT_RATE_{w}M")
    add(nm, "P4", "RATE", f"Engagement / exposure, {CH_LABEL.get(ch, 'without PulsePoint' if ch == 'XPP' else 'all paid media')}, "
                          f"last {w} months (NULL if no exposure; can exceed 1)", w)
for nm, d in [("DIGITAL_DAYS_SINCE_LAST_EXPOSURE", "paid-media exposure"), ("DIGITAL_DAYS_SINCE_LAST_ENGAGEMENT", "paid-media engagement"),
              ("DIGITAL_CH_3P_EMAIL_DAYS_SINCE_LAST_ENGAGEMENT", "3rd-party email engagement"),
              ("DIGITAL_CH_CUSTOM_DAYS_SINCE_LAST_ENGAGEMENT", "custom-placement engagement")]:   # P5 recency
    add(nm, "P5", "DAYS", f"Days from LOOKBACK_END_DATE to the last {d} in the last 12 months ({RECENCY_SENTINEL} if none)", 12)
for nm, d, w in [("DIGITAL_N_VENDORS_3M", "Distinct vendors with any activity", 3),
                 ("DIGITAL_N_VENDORS_12M", "Distinct vendors with any activity", 12),
                 ("DIGITAL_N_PLACEMENTS_3M", "Distinct mapped placements with any activity", 3),
                 ("DIGITAL_N_ACTIVE_MONTHS_6M", "Months with any paid-media activity", 6),
                 ("DIGITAL_N_ACTIVE_MONTHS_12M", "Months with any paid-media activity", 12)]:      # P6 diversity
    add(nm, "P6", "COUNT", f"{d}, last {w} months", w)
for nm, d in [("DIGITAL_EXPOSURE_3M_VS_PRIOR_3M", "exposure"), ("DIGITAL_ENGAGEMENT_3M_VS_PRIOR_3M", "engagement"),
              ("DIGITAL_XPP_EXPOSURE_3M_VS_PRIOR_3M", "exposure without PulsePoint")]:      # P7 trend
    add(nm, "P7", "TREND", f"(last 3M - prior 3M) / (last 3M + prior 3M), {d}; -1..1, 0 if both 0, "
                           f"NULL if DIGITAL_HISTORY_MONTHS < {TREND_MIN_HISTORY}", 6)
for i in INDICATIONS:                                                 # indication dimension (section 3.6)
    for m, w in INDICATION_FEATURES:
        add(f"DIGITAL_IND_{i}_{m}_{w}M", "IND", "COUNT", f"Paid media tagged {i} in the brand value, {m.lower()}, last {w} months", w)

META_COLS = ["HCP_COHORT_ID", "PROVIDER_ID", "COHORT", "DATASET", "PREDICTION_MONTH", "SVC_DT", "DIGITAL_HISTORY_MONTHS", "PULL_DT"]
FEATURES = [s["FEATURE"] for s in SPECS]
EXPECTED_COLUMNS = META_COLS + FEATURES
assert len(set(FEATURES)) == len(FEATURES), "duplicate feature names"

manifest = pd.DataFrame(SPECS).assign(DICT_VERSION=DICT_VERSION)
spark.createDataFrame(manifest.astype(object).where(manifest.notna(), None).astype(str)).write.mode("overwrite").parquet(MANIFEST_PATH)
print(f"Fixed schema: {len(FEATURES)} features")
print(manifest.groupby("FAMILY").size().to_string())

In [0]:
# Each cohort sees the 12 buckets ending at its END_BUCKET; K = 0 is the most recent month
cb = F.broadcast(cohorts.select("COHORT", "END_BUCKET", "LOOKBACK_END_DATE"))
j = (stage.join(cb, (stage.B <= cb.END_BUCKET) & (stage.B >= F.add_months(cb.END_BUCKET, -11)))
          .withColumn("K", F.months_between(col("END_BUCKET"), col("B")).cast("int"))
          .withColumn("ACT_ALL", col("EXP_ALL") + col("ENG_ALL")))

def wsum(c, w, lo=0):
    return F.sum(when((col("K") >= lo) & (col("K") < w), col(c)).otherwise(0.0))

aggs = []
for w in WINDOWS:
    for m, s in METRICS.items():
        aggs += [wsum(f"{s}_ALL", w).alias(f"DIGITAL_TOTAL_{m}_{w}M"), wsum(f"{s}_XPP", w).alias(f"DIGITAL_TOTAL_XPP_{m}_{w}M")]
        aggs += [wsum(f"{s}_CH_{ch}", w).alias(f"DIGITAL_CH_{ch}_{m}_{w}M") for ch in CHANNELS]
for p in PLACEMENTS:
    aggs += [wsum(f"{METRICS[m]}_PLC_{p}", w).alias(f"DIGITAL_PLC_{p}_{m}_{w}M") for m, w in PLACEMENT_FEATURES]
    aggs.append(F.sum(when(col("K") < 3, col(f"EXP_PLC_{p}") + col(f"ENG_PLC_{p}")).otherwise(0.0)).alias(f"_ACT_PLC_{p}_3M"))
for v in VENDORS:
    aggs += [wsum(f"ACT_V_{v}", 3).alias(f"_ACT_V_{v}_3M"), wsum(f"ACT_V_{v}", 12).alias(f"_ACT_V_{v}_12M")]
for i in INDICATIONS:
    aggs += [wsum(f"{METRICS[m]}_IND_{i}", w).alias(f"DIGITAL_IND_{i}_{m}_{w}M") for m, w in INDICATION_FEATURES]
aggs += [wsum("EXP_ALL", 6, 3).alias("_EXP_PRIOR_3M"), wsum("ENG_ALL", 6, 3).alias("_ENG_PRIOR_3M"),
         wsum("EXP_XPP", 6, 3).alias("_XPP_EXP_PRIOR_3M")]
aggs += [F.countDistinct(when((col("K") < w) & (col("ACT_ALL") > 0), col("B"))).alias(f"DIGITAL_N_ACTIVE_MONTHS_{w}M") for w in (6, 12)]
aggs += [F.max("LAST_EXP_DT").alias("_LAST_EXP_DT"), F.max("LAST_ENG_DT").alias("_LAST_ENG_DT"),
         F.max("LAST_ENG_DT_3P_EMAIL").alias("_LAST_ENG_DT_3P_EMAIL"), F.max("LAST_ENG_DT_CUSTOM").alias("_LAST_ENG_DT_CUSTOM"),
         F.first("LOOKBACK_END_DATE").alias("_END")]
act = j.groupBy("PROVIDER_ID", "COHORT").agg(*aggs)

# Leakage guard: no event after LOOKBACK_END_DATE
_leak = act.filter((col("_LAST_EXP_DT") > col("_END")) | (col("_LAST_ENG_DT") > col("_END"))).count()
assert _leak == 0, f"{_leak} rows with events after the cohort end"

def ratio(n, d):
    return when(col(d) > 0, col(n) / col(d))
def trend(a, b):
    return when((col(a) + col(b)) > 0, (col(a) - col(b)) / (col(a) + col(b))).otherwise(lit(0.0))
def days(c):
    return F.coalesce(F.datediff(col("_END"), col(c)), lit(RECENCY_SENTINEL))

derived = {
    "DIGITAL_ENGAGEMENT_RATE_3M":  ratio("DIGITAL_TOTAL_ENGAGEMENT_3M", "DIGITAL_TOTAL_EXPOSURE_3M"),
    "DIGITAL_ENGAGEMENT_RATE_6M":  ratio("DIGITAL_TOTAL_ENGAGEMENT_6M", "DIGITAL_TOTAL_EXPOSURE_6M"),
    "DIGITAL_ENGAGEMENT_RATE_12M": ratio("DIGITAL_TOTAL_ENGAGEMENT_12M", "DIGITAL_TOTAL_EXPOSURE_12M"),
    "DIGITAL_XPP_ENGAGEMENT_RATE_6M": ratio("DIGITAL_TOTAL_XPP_ENGAGEMENT_6M", "DIGITAL_TOTAL_XPP_EXPOSURE_6M"),
    **{f"DIGITAL_CH_{ch}_ENGAGEMENT_RATE_6M": ratio(f"DIGITAL_CH_{ch}_ENGAGEMENT_6M", f"DIGITAL_CH_{ch}_EXPOSURE_6M") for ch in CHANNELS},
    "DIGITAL_DAYS_SINCE_LAST_EXPOSURE": days("_LAST_EXP_DT"),
    "DIGITAL_DAYS_SINCE_LAST_ENGAGEMENT": days("_LAST_ENG_DT"),
    "DIGITAL_CH_3P_EMAIL_DAYS_SINCE_LAST_ENGAGEMENT": days("_LAST_ENG_DT_3P_EMAIL"),
    "DIGITAL_CH_CUSTOM_DAYS_SINCE_LAST_ENGAGEMENT": days("_LAST_ENG_DT_CUSTOM"),
    "DIGITAL_N_VENDORS_3M":  sum((col(f"_ACT_V_{v}_3M") > 0).cast("int") for v in VENDORS),
    "DIGITAL_N_VENDORS_12M": sum((col(f"_ACT_V_{v}_12M") > 0).cast("int") for v in VENDORS),
    "DIGITAL_N_PLACEMENTS_3M": sum((col(f"_ACT_PLC_{p}_3M") > 0).cast("int") for p in PLACEMENTS),
    "DIGITAL_EXPOSURE_3M_VS_PRIOR_3M": trend("DIGITAL_TOTAL_EXPOSURE_3M", "_EXP_PRIOR_3M"),
    "DIGITAL_ENGAGEMENT_3M_VS_PRIOR_3M": trend("DIGITAL_TOTAL_ENGAGEMENT_3M", "_ENG_PRIOR_3M"),
    "DIGITAL_XPP_EXPOSURE_3M_VS_PRIOR_3M": trend("DIGITAL_TOTAL_XPP_EXPOSURE_3M", "_XPP_EXP_PRIOR_3M"),
}
act = act.withColumns(derived).drop(*[c for c in act.columns if c.startswith("_")])

# Full grid: every universe HCP x every cohort; HCPs without paid media get 0 / sentinel / NULL rates
TREND_COLS = [s["FEATURE"] for s in SPECS if s["KIND"] == "TREND"]
COUNT_COLS = [s["FEATURE"] for s in SPECS if s["KIND"] in ("COUNT",)]
DAYS_COLS  = [s["FEATURE"] for s in SPECS if s["KIND"] == "DAYS"]
grid = universe.crossJoin(F.broadcast(cohorts.select("COHORT", "DATASET", "PREDICTION_MONTH", "LOOKBACK_END_DATE", "DIGITAL_HISTORY_MONTHS")))
df_final = (grid.join(act, ["PROVIDER_ID", "COHORT"], "left")
            .fillna(0.0, subset=COUNT_COLS)
            .fillna(0.0, subset=TREND_COLS)
            .fillna(RECENCY_SENTINEL, subset=DAYS_COLS)
            .withColumns({c: when(col("DIGITAL_HISTORY_MONTHS") >= TREND_MIN_HISTORY, col(c)) for c in TREND_COLS})
            .withColumns({c: col(c).cast("int") for c in DAYS_COLS + ["DIGITAL_N_VENDORS_3M", "DIGITAL_N_VENDORS_12M",
                          "DIGITAL_N_PLACEMENTS_3M", "DIGITAL_N_ACTIVE_MONTHS_6M", "DIGITAL_N_ACTIVE_MONTHS_12M"]})
            .withColumn("HCP_COHORT_ID", F.concat(col("PROVIDER_ID"), lit("_"), col("COHORT")))
            .withColumn("SVC_DT", col("LOOKBACK_END_DATE"))
            .withColumn("PULL_DT", lit(datetime.today().strftime("%Y_%m_%d")))
            .select(*EXPECTED_COLUMNS))

df_final.write.mode("overwrite").parquet(OUTPUT_PATH)
with open(SCHEMA_PATH, "w") as fh:
    json.dump({"DICT_VERSION": DICT_VERSION, "COLUMNS": EXPECTED_COLUMNS}, fh, indent=1)
df_final = spark.read.parquet(OUTPUT_PATH)
print(f"Written {df_final.count():,} rows x {len(df_final.columns)} columns -> {OUTPUT_PATH}")

In [0]:
qc = []
def check(name, ok, detail=""):
    qc.append((name, "PASS" if ok else "FAIL", str(detail)))

n_rows = df_final.count()
check("Schema = fixed list", df_final.columns == EXPECTED_COLUMNS, f"{len(df_final.columns)} columns")
check("Rows = universe x cohorts", n_rows == N_UNIVERSE * len(cohort_pd), f"{n_rows:,} vs {N_UNIVERSE * len(cohort_pd):,}")
check("HCP_COHORT_ID unique", df_final.select("HCP_COHORT_ID").distinct().count() == n_rows)

_nulls = df_final.select([F.sum(col(c).isNull().cast("int")).alias(c) for c in COUNT_COLS + DAYS_COLS]).first().asDict()
check("No NULL in count / days features", sum(_nulls.values()) == 0, {k: v for k, v in _nulls.items() if v})

nest = None
for m in METRICS:
    for a, b in [(1, 3), (3, 6), (6, 12)]:
        c = col(f"DIGITAL_TOTAL_{m}_{a}M") > col(f"DIGITAL_TOTAL_{m}_{b}M") + 1e-6
        nest = c if nest is None else (nest | c)
check("Windows nested (1M <= 3M <= 6M <= 12M)", df_final.filter(nest).count() == 0)

_ch_gap = df_final.filter(sum(col(f"DIGITAL_CH_{ch}_EXPOSURE_12M") for ch in CHANNELS) > col("DIGITAL_TOTAL_EXPOSURE_12M") + 1e-6).count()
check("Sum of channels <= total (12M)", _ch_gap == 0, _ch_gap)
check("XPP <= total (12M)", df_final.filter(col("DIGITAL_TOTAL_XPP_EXPOSURE_12M") > col("DIGITAL_TOTAL_EXPOSURE_12M") + 1e-6).count() == 0)
_bad_days = df_final.filter(~(col("DIGITAL_DAYS_SINCE_LAST_EXPOSURE").between(0, 366) |
                              (col("DIGITAL_DAYS_SINCE_LAST_EXPOSURE") == RECENCY_SENTINEL))).count()
check("Recency in [0, 366] or sentinel", _bad_days == 0, _bad_days)
_early = df_final.filter((col("DIGITAL_HISTORY_MONTHS") < TREND_MIN_HISTORY) & col("DIGITAL_EXPOSURE_3M_VS_PRIOR_3M").isNotNull()).count()
check("Trend NULL when history < 6", _early == 0, _early)
_rate_gt1 = df_final.filter(col("DIGITAL_ENGAGEMENT_RATE_6M") > 1).count()
check("Info: rows with engagement rate 6M > 1", True, _rate_gt1)

# ---- Independent recompute from raw MMX (date filter on DATE_ACT, raw channel names, no stage / bucket logic) ----
last = cohort_pd.iloc[-1]
END = str(last.LOOKBACK_END_DATE)
sample = (df_final.filter((col("COHORT") == last.COHORT) & (col("DIGITAL_TOTAL_EXPOSURE_3M") > 0))
          .orderBy(F.rand(42)).limit(15)
          .select("PROVIDER_ID", "DIGITAL_TOTAL_EXPOSURE_3M", "DIGITAL_TOTAL_ENGAGEMENT_12M", "DIGITAL_CH_BANNER_EXPOSURE_3M",
                  "DIGITAL_CH_CUSTOM_ENGAGEMENT_6M"))
raw = (spark.table(MMX_TABLE)
       .filter(F.upper(col("BRND_NM")).like("%KERENDIA%") & F.trim(col("NPI_NUM")).rlike("^[0-9]{10}$"))
       .withColumn("RCH", F.upper(F.trim(col("CHANNEL"))))
       .filter(col("RCH").isin("BANNER", "3RD PARTY EMAIL", "CUSTOM"))
       .filter((col("DATE_ACT") > F.add_months(F.lit(END).cast("date"), -12)) & (col("DATE_ACT") <= F.lit(END).cast("date")))
       .join(npi_map, F.trim(col("NPI_NUM")) == npi_map.NPI_NUMBER)
       .join(sample.select("PROVIDER_ID"), "PROVIDER_ID"))
def in_last(w):
    return col("DATE_ACT") > F.add_months(F.lit(END).cast("date"), -w)
ex, en = F.coalesce(col("EXPOSURE").cast("double"), lit(0.0)), F.coalesce(col("TOTAL_ENGAGEMENT").cast("double"), lit(0.0))
recomp = raw.groupBy("PROVIDER_ID").agg(
    F.sum(when(in_last(3), ex).otherwise(0.0)).alias("R_TOTAL_EXPOSURE_3M"),
    F.sum(en).alias("R_TOTAL_ENGAGEMENT_12M"),
    F.sum(when(in_last(3) & (col("RCH") == "BANNER"), ex).otherwise(0.0)).alias("R_CH_BANNER_EXPOSURE_3M"),
    F.sum(when(in_last(6) & (col("RCH") == "CUSTOM"), en).otherwise(0.0)).alias("R_CH_CUSTOM_ENGAGEMENT_6M"))
cmp = sample.join(recomp, "PROVIDER_ID", "left").fillna(0.0).toPandas()
pairs = [("DIGITAL_TOTAL_EXPOSURE_3M", "R_TOTAL_EXPOSURE_3M"), ("DIGITAL_TOTAL_ENGAGEMENT_12M", "R_TOTAL_ENGAGEMENT_12M"),
         ("DIGITAL_CH_BANNER_EXPOSURE_3M", "R_CH_BANNER_EXPOSURE_3M"), ("DIGITAL_CH_CUSTOM_ENGAGEMENT_6M", "R_CH_CUSTOM_ENGAGEMENT_6M")]
n_match = sum(int(abs(r[a] - r[b]) < 1e-6) for _, r in cmp.iterrows() for a, b in pairs)
check(f"Raw MMX recompute ({len(cmp)} HCPs x {len(pairs)} features, cohort {last.COHORT})", n_match == len(cmp) * len(pairs),
      f"{n_match}/{len(cmp) * len(pairs)} match")

qc_pd = pd.DataFrame(qc, columns=["CHECK", "RESULT", "DETAIL"])
display(qc_pd)
if (qc_pd.RESULT == "FAIL").any():
    display(cmp)

In [0]:
display(df_final.groupBy("COHORT", "DATASET", "SVC_DT", "DIGITAL_HISTORY_MONTHS").agg(
    F.count("*").alias("HCPS"),
    F.sum((col("DIGITAL_TOTAL_EXPOSURE_3M") > 0).cast("int")).alias("HCPS_EXPOSED_3M"),
    F.sum("DIGITAL_TOTAL_EXPOSURE_3M").alias("EXPOSURE_3M"),
    F.sum("DIGITAL_TOTAL_XPP_EXPOSURE_3M").alias("EXPOSURE_XPP_3M"),
    F.sum("DIGITAL_TOTAL_ENGAGEMENT_3M").alias("ENGAGEMENT_3M"),
    F.avg("DIGITAL_N_VENDORS_12M").alias("AVG_VENDORS_12M"))
  .orderBy(F.expr("int(substring(COHORT, 2))")))